# BERTScore — Ground Truth × Gerações Gherkin (JSON)

Notebook adaptado para o mesmo formato de dados utilizado no cálculo da **Distância de Manhattan**:

- **Ground truth**: JSON com `cases`, em que cada caso possui `case_id`, `original_case`, `reference_id` e `gherkin`.
- **Gerações**: JSON com metadados `model`, `technique`, `number_of_executions` e, para cada caso, uma lista `generations`.
- A associação entre referência e geração é feita por **`case_id`**, e não pela posição do caso no arquivo.
- O cálculo mantém a configuração do notebook BERTScore anterior: `lang='pt'` e `rescale_with_baseline=True`.
- O notebook aceita **1 ground truth e 1 ou mais arquivos de gerações**.
- Precision, Recall e F1 são exportados em **percentual (0–100)**; o ranking de cada caso é feito pela **maior F1**.
- O cálculo é realizado em lote para evitar uma chamada ao BERTScore para cada par individualmente.
- Ao final, é gerado um **CSV detalhado** com todas as comparações e rankings.

> **Interpretação:** quanto maior o BERTScore F1, maior a similaridade semântica entre o cenário de referência e o cenário gerado.


In [ ]:
# ============================================================
# 1. INSTALAÇÃO DA DEPENDÊNCIA
# ============================================================

!pip -q install bert-score


In [ ]:
# ============================================================
# 2. IMPORTAÇÕES E CONFIGURAÇÃO
# ============================================================

import json
import re
import warnings

import pandas as pd
import torch
from IPython.display import display
from bert_score import score as bertscore_score

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)


BERTSCORE_LANG = "pt"
RESCALE_WITH_BASELINE = True

# O notebook usa GPU quando disponível e funciona em CPU caso contrário.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Ajuste se houver pouca memória de GPU/CPU.
BERTSCORE_BATCH_SIZE = 16 if DEVICE == "cuda" else 8

# Se True, o CSV será baixado automaticamente ao final no Google Colab.
BAIXAR_CSV_AUTOMATICAMENTE = True

print(f"Dispositivo selecionado: {DEVICE}")
print(f"Batch size: {BERTSCORE_BATCH_SIZE}")


Dispositivo selecionado: cuda
Batch size: 16


In [ ]:
# ============================================================
# 3. UPLOAD E IDENTIFICAÇÃO AUTOMÁTICA DOS JSONs
# ============================================================

def carregar_json_bytes(nome_arquivo, conteudo):
    # Aceita UTF-8 com ou sem BOM.
    try:
        texto = conteudo.decode("utf-8-sig")
        return json.loads(texto)
    except Exception as e:
        raise ValueError(f"Não foi possível ler '{nome_arquivo}' como JSON: {e}") from e


def classificar_json(nome_arquivo, dados):
    # Classifica como ground_truth, geracoes ou desconhecido.
    if not isinstance(dados, dict):
        return "desconhecido"

    casos = dados.get("cases")
    if not isinstance(casos, list):
        return "desconhecido"

    if dados.get("is_reference_base") is True:
        return "ground_truth"

    if casos:
        primeiro = casos[0]
        if isinstance(primeiro, dict):
            if "generations" in primeiro:
                return "geracoes"
            if "reference_id" in primeiro and "gherkin" in primeiro:
                return "ground_truth"

    return "desconhecido"


def processar_upload(uploaded):
    arquivos = []

    for nome, conteudo in uploaded.items():
        if not nome.lower().endswith(".json"):
            print(f"⚠ Ignorado (não é JSON): {nome}")
            continue

        dados = carregar_json_bytes(nome, conteudo)
        tipo = classificar_json(nome, dados)
        arquivos.append({"nome": nome, "tipo": tipo, "dados": dados})

    return arquivos


try:
    from google.colab import files
except ImportError as e:
    raise RuntimeError(
        "Este notebook foi preparado para upload interativo no Google Colab. "
        "Execute-o no Colab ou adapte esta célula para leitura local."
    ) from e


# ------------------------------------------------------------
# ETAPA 1 — Ground truth
# ------------------------------------------------------------
print("ETAPA 1/2 — Envie o arquivo JSON do ground truth:")
upload_gt = files.upload()
arquivos_json = processar_upload(upload_gt)

ground_truths = [a for a in arquivos_json if a["tipo"] == "ground_truth"]
arquivos_geracoes = [a for a in arquivos_json if a["tipo"] == "geracoes"]
desconhecidos = [a["nome"] for a in arquivos_json if a["tipo"] == "desconhecido"]

if desconhecidos:
    print("⚠ JSON(s) com estrutura não reconhecida:", desconhecidos)

if len(ground_truths) != 1:
    raise ValueError(
        f"É necessário exatamente 1 ground truth. Foram identificados {len(ground_truths)}. "
        "Verifique se o arquivo possui 'is_reference_base': true ou casos com "
        "'reference_id' e 'gherkin'."
    )

ground_truth_nome = ground_truths[0]["nome"]
ground_truth = ground_truths[0]["dados"]

print(f"\n✓ Ground truth identificado: {ground_truth_nome}")


# ------------------------------------------------------------
# ETAPA 2 — Gerações
# ------------------------------------------------------------
# Se gerações forem enviadas junto com o ground truth, elas são aproveitadas.
# Caso contrário, abre um segundo seletor.
if not arquivos_geracoes:
    print("\nETAPA 2/2 — Agora envie um ou mais JSONs de gerações:")
    upload_gen = files.upload()
    novos_arquivos = processar_upload(upload_gen)

    novos_ground_truths = [a for a in novos_arquivos if a["tipo"] == "ground_truth"]
    if novos_ground_truths:
        print(
            "⚠ Ground truth adicional ignorado na etapa de gerações:",
            [a["nome"] for a in novos_ground_truths]
        )

    novos_desconhecidos = [
        a["nome"] for a in novos_arquivos if a["tipo"] == "desconhecido"
    ]
    if novos_desconhecidos:
        print("⚠ JSON(s) com estrutura não reconhecida:", novos_desconhecidos)

    arquivos_geracoes.extend(
        a for a in novos_arquivos if a["tipo"] == "geracoes"
    )

if not arquivos_geracoes:
    raise ValueError(
        "Nenhum arquivo de gerações foi identificado. Os arquivos de gerações "
        "devem possuir 'cases' e, dentro de cada caso, a chave 'generations'."
    )

print(f"\n✓ Arquivos de gerações identificados: {len(arquivos_geracoes)}")
for arq in arquivos_geracoes:
    dados = arq["dados"]
    print(
        f"  - {arq['nome']} | modelo={dados.get('model')} | "
        f"técnica={dados.get('technique')} | "
        f"execuções declaradas={dados.get('number_of_executions')}"
    )


ETAPA 1/2 — Envie o arquivo JSON do ground truth:


Saving base_referencia_gherkin.json to base_referencia_gherkin.json

✓ Ground truth identificado: base_referencia_gherkin.json

ETAPA 2/2 — Agora envie um ou mais JSONs de gerações:


Saving geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json to geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json

✓ Arquivos de gerações identificados: 1
  - geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json | modelo=ibm-granite/granite-4.1-8b | técnica=few-shot | execuções declaradas=10


In [ ]:
# ============================================================
# 4. VALIDAÇÃO DOS DADOS
# ============================================================

def indexar_ground_truth(dados_ground_truth):
    refs = {}
    duplicados = []

    for caso in dados_ground_truth.get("cases", []):
        case_id = caso.get("case_id")

        if not case_id:
            continue

        if case_id in refs:
            duplicados.append(case_id)

        refs[case_id] = caso

    if duplicados:
        raise ValueError(
            f"case_id duplicado(s) no ground truth: {sorted(set(duplicados))}"
        )

    return refs


referencias = indexar_ground_truth(ground_truth)
avisos_validacao = []

for arq in arquivos_geracoes:
    nome = arq["nome"]
    dados = arq["dados"]
    ids_arquivo = []
    declaradas = dados.get("number_of_executions")

    for caso in dados.get("cases", []):
        case_id = caso.get("case_id")
        ids_arquivo.append(case_id)

        if case_id not in referencias:
            avisos_validacao.append(
                f"{nome}: {case_id} existe nas gerações, mas não no ground truth."
            )
            continue

        original_ref = referencias[case_id].get("original_case")
        original_gen = caso.get("original_case")

        if (
            original_ref is not None
            and original_gen is not None
            and original_ref != original_gen
        ):
            avisos_validacao.append(
                f"{nome}: original_case divergente em {case_id}."
            )

        generations = caso.get("generations", [])

        if declaradas is not None and len(generations) != declaradas:
            avisos_validacao.append(
                f"{nome}: {case_id} possui {len(generations)} gerações, "
                f"mas o arquivo declara {declaradas}."
            )

        execucoes = [g.get("execution") for g in generations]
        execucoes_validas = [e for e in execucoes if e is not None]

        if len(execucoes_validas) != len(set(execucoes_validas)):
            avisos_validacao.append(
                f"{nome}: há números de execução duplicados em {case_id}."
            )

    ids_ref = set(referencias)
    ids_gen = set(ids_arquivo)

    ausentes = sorted(ids_ref - ids_gen)
    if ausentes:
        avisos_validacao.append(
            f"{nome}: {len(ausentes)} case_id(s) do ground truth não aparecem nas gerações. "
            f"Exemplos: {ausentes[:10]}"
        )

print(f"Casos no ground truth: {len(referencias)}")

if avisos_validacao:
    print(f"\n⚠ Foram encontrados {len(avisos_validacao)} aviso(s) de validação:")
    for aviso in avisos_validacao:
        print(" -", aviso)
else:
    print("\n✓ Estrutura validada sem avisos.")


Casos no ground truth: 259

✓ Estrutura validada sem avisos.


In [ ]:
# ============================================================
# 5. PREPARAÇÃO DAS COMPARAÇÕES
# ============================================================

comparacoes = []

for arq in arquivos_geracoes:
    nome_geracoes = arq["nome"]
    dados = arq["dados"]

    modelo = dados.get("model", "")
    tecnica = dados.get("technique", "")
    execucoes_declaradas = dados.get("number_of_executions")

    for caso_gerado in dados.get("cases", []):
        case_id = caso_gerado.get("case_id")
        referencia = referencias.get(case_id)

        if referencia is None:
            continue

        gherkin_ref = referencia.get("gherkin", "")

        for geracao in caso_gerado.get("generations", []):
            gherkin_gerado = geracao.get("gherkin", "")

            comparacoes.append({
                "arquivo_ground_truth": ground_truth_nome,
                "arquivo_geracoes": nome_geracoes,
                "modelo": modelo,
                "tecnica": tecnica,
                "execucoes_declaradas": execucoes_declaradas,
                "case_id": case_id,
                "source_id": referencia.get("source_id"),
                "source_line": referencia.get("source_line"),
                "original_case": referencia.get("original_case"),
                "reference_id": referencia.get("reference_id"),
                "generation_id": geracao.get("generation_id"),
                "execucao": geracao.get("execution"),
                "gherkin_ground_truth": "" if gherkin_ref is None else str(gherkin_ref),
                "gherkin_gerado": "" if gherkin_gerado is None else str(gherkin_gerado),
            })

if not comparacoes:
    raise ValueError("Nenhuma comparação pôde ser preparada.")

print(f"✓ Comparações preparadas: {len(comparacoes):,}")


✓ Comparações preparadas: 2,590


In [ ]:
# ============================================================
# 6. CÁLCULO DO BERTSCORE
# ============================================================

# No BERTScore:
#   cands = textos gerados
#   refs  = textos de referência
#
# Mantemos lang='pt' e rescale_with_baseline=True, como no notebook anterior.
candidatos = [r["gherkin_gerado"] for r in comparacoes]
referencias_texto = [r["gherkin_ground_truth"] for r in comparacoes]

print(
    f"Calculando BERTScore para {len(comparacoes):,} pares "
    f"em {DEVICE.upper()}..."
)

P, R, F1 = bertscore_score(
    candidatos,
    referencias_texto,
    lang=BERTSCORE_LANG,
    rescale_with_baseline=RESCALE_WITH_BASELINE,
    batch_size=BERTSCORE_BATCH_SIZE,
    device=DEVICE,
    verbose=True,
)

precision_percentual = (P.detach().cpu().numpy() * 100).tolist()
recall_percentual = (R.detach().cpu().numpy() * 100).tolist()
f1_percentual = (F1.detach().cpu().numpy() * 100).tolist()

resultados = []

for registro, precision, recall, f1 in zip(
    comparacoes,
    precision_percentual,
    recall_percentual,
    f1_percentual
):
    linha = dict(registro)

    # Mantém o padrão anterior de apresentar a pontuação em percentual,
    # mas preserva mais casas no CSV para não perder precisão estatística.
    linha["bertscore_precision"] = float(precision)
    linha["bertscore_recall"] = float(recall)
    linha["bertscore_f1"] = float(f1)

    resultados.append(linha)

df_resultados = pd.DataFrame(resultados)

chaves_ranking = ["arquivo_geracoes", "modelo", "tecnica", "case_id"]

# No BERTScore, maior F1 = melhor resultado.
df_resultados = df_resultados.sort_values(
    chaves_ranking + ["bertscore_f1", "execucao"],
    ascending=[True, True, True, True, False, True],
    kind="stable",
    na_position="last"
).reset_index(drop=True)

# Ranking sequencial após ordenar pela maior F1.
# Assim como no notebook de Manhattan, empates ocupam posições sucessivas.
df_resultados["ranking_no_caso"] = (
    df_resultados.groupby(chaves_ranking, dropna=False).cumcount() + 1
)

colunas = [
    "modelo",
    "tecnica",
    "case_id",
    "source_id",
    "original_case",
    "execucao",
    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1",
    "ranking_no_caso",
    "generation_id",
    "reference_id",
    "gherkin_ground_truth",
    "gherkin_gerado",
    "execucoes_declaradas",
    "arquivo_ground_truth",
    "arquivo_geracoes",
]

df_resultados = df_resultados[colunas]

print(f"\n✓ Comparações calculadas: {len(df_resultados):,}")
print(f"✓ Casos avaliados: {df_resultados['case_id'].nunique():,}")


Calculando BERTScore para 2,590 pares em CUDA...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/164 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/162 [00:00<?, ?it/s]

done in 14.59 seconds, 177.48 sentences/sec

✓ Comparações calculadas: 2,590
✓ Casos avaliados: 259


In [ ]:
# ============================================================
# 7. TABELAS ORGANIZADAS
# ============================================================

df_resumo_geral = (
    df_resultados
    .groupby(["arquivo_geracoes", "modelo", "tecnica"], dropna=False)
    .agg(
        casos=("case_id", "nunique"),
        comparacoes=("bertscore_f1", "count"),
        f1_media=("bertscore_f1", "mean"),
        f1_mediana=("bertscore_f1", "median"),
        desvio_padrao=("bertscore_f1", "std"),
        f1_minima=("bertscore_f1", "min"),
        f1_maxima=("bertscore_f1", "max"),
    )
    .reset_index()
)

print("RESUMO GERAL")
display(
    df_resumo_geral.style.format({
        "f1_media": "{:.2f}",
        "f1_mediana": "{:.2f}",
        "desvio_padrao": "{:.2f}",
        "f1_minima": "{:.2f}",
        "f1_maxima": "{:.2f}",
    })
)

df_resumo_casos = (
    df_resultados
    .groupby(
        ["arquivo_geracoes", "modelo", "tecnica", "case_id", "original_case"],
        dropna=False
    )
    .agg(
        execucoes_avaliadas=("execucao", "count"),
        f1_media=("bertscore_f1", "mean"),
        f1_mediana=("bertscore_f1", "median"),
        desvio_padrao=("bertscore_f1", "std"),
        melhor_f1=("bertscore_f1", "max"),
        pior_f1=("bertscore_f1", "min"),
    )
    .reset_index()
    .sort_values(["modelo", "tecnica", "case_id"])
)

print("\nRESUMO POR CASO — primeiras 30 linhas")
display(
    df_resumo_casos.head(30).style.format({
        "f1_media": "{:.2f}",
        "f1_mediana": "{:.2f}",
        "desvio_padrao": "{:.2f}",
        "melhor_f1": "{:.2f}",
        "pior_f1": "{:.2f}",
    })
)

print("\nCOMPARAÇÕES DETALHADAS — primeiras 50 linhas")
colunas_visualizacao = [
    "modelo",
    "tecnica",
    "case_id",
    "original_case",
    "execucao",
    "bertscore_precision",
    "bertscore_recall",
    "bertscore_f1",
    "ranking_no_caso",
]

display(
    df_resultados[colunas_visualizacao]
    .head(50)
    .style
    .format({
        "bertscore_precision": "{:.2f}",
        "bertscore_recall": "{:.2f}",
        "bertscore_f1": "{:.2f}",
    })
)


RESUMO GERAL


,arquivo_geracoes,modelo,tecnica,casos,comparacoes,f1_media,f1_mediana,desvio_padrao,f1_minima,f1_maxima
0,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,259,2590,47.08,47.19,7.65,16.37,69.89



RESUMO POR CASO — primeiras 30 linhas


,arquivo_geracoes,modelo,tecnica,case_id,original_case,execucoes_avaliadas,f1_media,f1_mediana,desvio_padrao,melhor_f1,pior_f1
0,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,10,41.38,41.99,2.68,44.67,36.56
1,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1002,Cadastrar transação com campos inválidos,10,51.78,52.09,3.66,55.19,45.35
2,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1006,Verificar se todos os dados da transação estão sendo exibidos,10,50.20,50.89,2.23,53.17,46.26
3,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1007,Verificar inserção de link da transação inexistente,10,46.33,47.52,3.23,50.54,41.02
4,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_101,Validar resultado de consulta de Matéria-Prima vazia,10,51.33,51.29,1.29,53.18,49.48
5,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1030,Cadastar categoria com sucesso,10,47.64,47.72,1.26,49.65,45.17
6,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1033,Cadastrar categoria com campos inválidos,10,52.30,52.21,3.59,58.58,47.49
7,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1035,Editar categoria deixando os campos obrigatórios do formulário em branco,10,47.34,47.34,3.78,53.31,43.24
8,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1060,Cancelar cadastro de conta com sucesso,10,37.99,38.17,1.31,40.29,35.84
9,geracoes_gherkin_ibm-granite-granite-4.1-8b_few-shot.json,ibm-granite/granite-4.1-8b,few-shot,TC_1061,Cadastrar conta com campos obrigatórios não preenchidos,10,56.61,56.07,1.81,59.21,54.12



COMPARAÇÕES DETALHADAS — primeiras 50 linhas


,modelo,tecnica,case_id,original_case,execucao,bertscore_precision,bertscore_recall,bertscore_f1,ranking_no_caso
0,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,6,34.32,55.97,44.67,1
1,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,9,34.32,55.97,44.67,2
2,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,1,34.95,51.76,43.10,3
3,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,10,35.14,50.95,42.83,4
4,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,4,33.06,52.28,42.31,5
5,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,5,31.61,52.62,41.67,6
6,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,3,31.14,50.03,40.24,7
7,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,7,33.25,44.68,38.89,8
8,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,8,33.25,44.68,38.89,9
9,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,2,30.40,42.93,36.56,10


In [ ]:
# ============================================================
# 8. CONSULTA RÁPIDA DE UM CASO
# ============================================================

def visualizar_caso(case_id):
    # Exibe todas as execuções de um case_id, da maior para a menor F1.
    recorte = df_resultados[df_resultados["case_id"] == case_id].copy()

    if recorte.empty:
        print(f"Nenhum resultado encontrado para {case_id}.")
        return

    colunas = [
        "modelo",
        "tecnica",
        "case_id",
        "original_case",
        "execucao",
        "bertscore_precision",
        "bertscore_recall",
        "bertscore_f1",
        "ranking_no_caso",
        "gherkin_ground_truth",
        "gherkin_gerado",
    ]

    display(
        recorte[colunas]
        .sort_values(
            ["modelo", "tecnica", "bertscore_f1", "execucao"],
            ascending=[True, True, False, True]
        )
        .style
        .format({
            "bertscore_precision": "{:.2f}",
            "bertscore_recall": "{:.2f}",
            "bertscore_f1": "{:.2f}",
        })
    )


primeiro_case_id = df_resultados["case_id"].iloc[0]
print(f"Exemplo de consulta: {primeiro_case_id}")
visualizar_caso(primeiro_case_id)

# Para consultar outro:
# visualizar_caso("TC_261")


Exemplo de consulta: TC_1


,modelo,tecnica,case_id,original_case,execucao,bertscore_precision,bertscore_recall,bertscore_f1,ranking_no_caso,gherkin_ground_truth,gherkin_gerado
0,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,6,34.32,55.97,44.67,1,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação indicando que a entidade foi cadastrada com sucesso And o novo registro deve aparecer na lista de entidades existentes"
1,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,9,34.32,55.97,44.67,2,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação indicando que a entidade foi cadastrada com sucesso And o novo registro deve aparecer na lista de entidades existentes"
2,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,1,34.95,51.76,43.10,3,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve confirmar a criação da entidade com uma mensagem de sucesso And o novo registro deve aparecer na lista de entidades cadastradas"
3,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,10,35.14,50.95,42.83,4,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro da entidade And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão de salvar Then o sistema deve validar os dados e exibir uma mensagem de sucesso And a entidade deve ser armazenada no banco de dados
4,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,4,33.06,52.28,42.31,5,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidade And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação ""Entidade cadastrada com sucesso"" And o novo registro deve aparecer na lista de entidades existentes"
5,ibm-granite/granite-4.1-8b,few-shot,TC_1,Cadastrar Entidade com sucesso,5,31.61,52.62,41.67,6,Scenario: Cadastrar Entidade com sucesso Given que o usuário está na funcionalidade de cadastro de entidade When ele preenche os dados e confirma o cadastro Then a entidade é cadastrada com sucesso,"Scenario: Cadastrar Entidade com sucesso Given o usuário está na página de cadastro de entidades And o usuário preenche todos os campos obrigatórios com dados válidos When o usuário clica no botão ""Salvar"" Then o sistema deve exibir uma mensagem de confirmação 

In [ ]:
# ============================================================
# 9. EXPORTAÇÃO DO CSV
# ============================================================

def slug(texto):
    texto = str(texto or "").strip().lower()
    texto = re.sub(r"[^a-z0-9._-]+", "-", texto)
    texto = re.sub(r"-+", "-", texto).strip("-")
    return texto or "sem-identificacao"


metadados_unicos = (
    df_resultados[["modelo", "tecnica"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

if len(metadados_unicos) == 1:
    modelo = metadados_unicos.loc[0, "modelo"]
    tecnica = metadados_unicos.loc[0, "tecnica"]
    nome_csv = f"metricas_bertscore_{slug(modelo)}_{slug(tecnica)}.csv"
else:
    nome_csv = "metricas_bertscore_multiplos_modelos_tecnicas.csv"

# UTF-8 com BOM facilita a abertura correta de acentos no Excel.
df_resultados.to_csv(nome_csv, index=False, encoding="utf-8-sig")

print(f"✓ CSV gerado: {nome_csv}")
print(f"✓ Linhas exportadas: {len(df_resultados):,}")

if BAIXAR_CSV_AUTOMATICAMENTE:
    try:
        from google.colab import files
        files.download(nome_csv)
    except Exception as e:
        print(f"Download automático não realizado: {e}")


✓ CSV gerado: metricas_bertscore_ibm-granite-granite-4.1-8b_few-shot.csv
✓ Linhas exportadas: 2,590


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>